# Gemma4 Router - Merge LoRA sang GGUF (Colab)
## Merge LoRA adapter da fine-tune thanh GGUF Q4_K_M

**Muc dich:** Merge LoRA adapter (da fine-tune tren Kaggle) vao base model va export sang GGUF.
**Khong train lai** -- chi merge + convert, chay ~5-10 phut tren GPU T4.
**Output:** GGUF Q4_K_M ~1.5-3GB, tai ve server de chay Ollama.

**Setup:**
1. Colab Settings -> Accelerator -> GPU
2. Colab Settings -> Internet -> On
3. Them secret `HF_TOKEN` trong Secrets panel (ben trai)
4. Chay lan luot Cell 1-6

In [ ]:
# Cell 1: Cai dat dependencies
!pip install unsloth --quiet
!pip install "gguf>=0.10.0" --quiet
from google.colab import userdata
import os
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
print(f"HF_TOKEN: {'OK' if HF_TOKEN else 'THIEU'}")
print("Dependencies da cai dat!")

In [ ]:
# Cell 2: Load base model + LoRA adapter tu HF repo
# LoRA nam trong subfolder: models/20260515_182858/lora/
from unsloth import FastLanguageModel
from huggingface_hub import snapshot_download
import torch

# Buoc 1: Tai base model (giong model ma LoRA duoc train tren do)
print("Buoc 1: Tai base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-e2b-unsloth-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)
print("Base model da tai xong!")

# Buoc 2: Tai LoRA adapter tu subfolder tren HF repo
print("Buoc 2: Tai LoRA adapter...")
lora_local_dir = "/content/lora"
snapshot_download(
    repo_id="diepxuan/dx-gemma4",
    local_dir=lora_local_dir,
    allow_patterns="models/20260515_182858/lora/*",
    token=HF_TOKEN,
)
print(f"LoRA da tai ve: {lora_local_dir}")

# Buoc 3: Load LoRA vao model
print("Buoc 3: Load LoRA adapter vao model...")
model.load_adapter(f"{lora_local_dir}/models/20260515_182858/lora", adapter_name="router")
model.set_adapter("router")
print("LoRA adapter da load thanh cong!")

In [ ]:
# Cell 3: Setup chat template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)
print("Chat template da setup!")

In [ ]:
# Cell 4: Export sang GGUF (merge LoRA + quantize Q4_K_M)
from pathlib import Path
import traceback

output_dir = "/content/gemma4-router-gguf"
Path(output_dir).mkdir(exist_ok=True)

print(f"Dang export GGUF Q4_K_M -> {output_dir}")
print("Buoc nay mat 5-10 phut, vui long doi...")

try:
    model.save_pretrained_gguf(
        output_dir,
        tokenizer,
        quantization_method="q4_k_m",
    )
    gguf_files = list(Path(output_dir).glob("*.gguf"))
    if gguf_files:
        for f in gguf_files:
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"GGUF: {f.name} ({size_mb:.0f} MB)")
        print("Export GGUF hoan thanh!")
    else:
        print("KHONG TIM THAY FILE GGUF. Kiem tra log tren.")
except Exception:
    print("Export GGUF that bai:")
    traceback.print_exc()
    raise

In [ ]:
# Cell 5: Test nhanh inference
FastLanguageModel.for_inference(model)

test_prompt = "Giup toi viet mot REST API voi Flask de quan ly nguoi dung"
messages = [{"role": "user", "content": test_prompt}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.1,
    top_p=0.95,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Input: {test_prompt}")
print(f"Output:\n{response}")

In [ ]:
# Cell 6: Download GGUF ve may
from google.colab import files

gguf_files = list(Path("/content/gemma4-router-gguf").glob("*.gguf"))
if gguf_files:
    print(f"Tim thay {len(gguf_files)} file GGUF:")
    for f in gguf_files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name} ({size_mb:.0f} MB)")
    print("\nDang tai xuong...")
    files.download(str(gguf_files[0]))
else:
    print("KHONG TIM THAY FILE GGUF")
    print("Chay Cell 4 truoc, sau do chay Cell nay.")

## Sau khi download xong

Tren server local:
```bash
# Dat file GGUF vao thu muc:
/root/dx-gemma4-artifacts/20260515_182858/base_gguf/

# Cap nhat Modelfile tro den GGUF moi, roi build:
cd /root/dx-gemma4/ollama
ollama create dx-gemma4-router -f Modelfile-no-lora

# Test:
ollama run dx-gemma4-router
```